In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# magic autoreload
%load_ext autoreload
%autoreload 2
from utils import *
from svd_imputer import Imputer

In [ ]:
datadir = "data"
df_dict = {}
for f in os.listdir(datadir):
    if f.startswith("serie"):
        continue
    df = pd.read_csv(os.path.join(datadir, f))
    df.columns = [c.lower() for c in df.columns]
    df.dropna(how="all", inplace=True)
    df = df.loc[df.date!="NaN-NaN-NaN NaN:NaN:NaN"]
    df.set_index("date", inplace=True)
    # Strip whitespace from index
    df.index = df.index.str.strip()
    try:
        df.index = pd.to_datetime(df.index, format="%Y-%m-%d %H:%M:%S")
    except ValueError:
        try:
            df.index = pd.to_datetime(df.index, format="%d/%m/%Y %H:%M")
        except ValueError:
            # If both fail, use infer_datetime_format
            df.index = pd.to_datetime(df.index, infer_datetime_format=True)
    df = df.squeeze()
    if isinstance(df, pd.DataFrame):
        df = df["wl_corr"]
    df.dropna(inplace=True)
    df.name = f.split(".")[0].lower()
    df_dict[df.name] = df
    print(f"{f}: {df.shape}")
data = pd.DataFrame(df_dict)

In [ ]:
#data.index = pd.to_datetime(data.index, format="mixed")
data = data.sort_index()

In [ ]:
data = data.drop(columns=["eramag03","eramag05","eramcus","eramag09w"])

In [ ]:
data_day = data.resample("D").mean().copy()
data_day.dropna(how="all", inplace=True)
data_day.plot(figsize=(6, 4), marker=".", markersize=1,lw=1,alpha=0.5)

In [ ]:
from svd_imputer.preprocessing import create_derivative_augmented_matrix, create_symmetric_augmented_matrix, create_asymmetric_augmented_matrix

data_expanded = data_day.copy()
data_expanded = create_derivative_augmented_matrix(data_expanded)
data_expanded.shape,data.shape

data_expanded.head()

In [ ]:
imputer = Imputer(data_expanded, rank=3,tol=1e-3)      # validate_dataframe() + preprocessing ONCE
imputer.fit()                        # Pure computation, cached SVD components
results = imputer.transform()        # Uses cached data + SVD components  
unc = imputer.estimate_uncertainty()  # Uses cached data
new_projected = imputer.project_data(data_expanded)    # NEW: SVD projection
new_reconstructed = imputer.reconstruct_data(data_expanded)  # NEW: SVD reconstruction

In [ ]:
#imputer = Imputer(rank=3,tol=1e-3,  verbose=True)
#df_imputed, unc = imputer.fit_transform(data_expanded,
#                                        return_uncertainty=True,
#                                        #uncertainty_method='monte_carlo',
#                                        #n_bootstrap=100,
#                                        n_repeats=50,
#                                        mask_strategy='random',
#                                        frac=.1)
#df_lower, df_upper = imputer.get_confidence_intervals(df_imputed, unc)
#

In [ ]:
data_ = data_day.dropna(how="all")

In [ ]:
mcres = np.array(unc['raw_imputed'])
mcres[:,:,:].shape

fig,axs = plt.subplots(2,1, figsize=(7,6),sharex=True)

for e,ax in enumerate(axs):
    ax.plot(data_.index,data_.iloc[:,e],'r.',zorder=1,lw=1)
    ax.axhline(data_.iloc[:,e].mean(),c='k',linestyle='--')

    ax.plot(results.index,results.iloc[:,e],'b-',zorder=1,lw=.5)
    [ax.plot(results.index, mcres[i,:,e],c="0.5",alpha=0.3,zorder=0) for i in range(mcres.shape[0])]
    ax.set_xlim(pd.to_datetime("01-01-1995"),
                pd.to_datetime("01-01-2015"))
    #ax.set_ylim(data_.iloc[:,e].min()-1,data_.iloc[:,e].max()+1)
fig.tight_layout()